# Figure 6 | Aβ-associated NVU gene changes

Cleaned notebook for the public NVU AD spatial transcriptomics repository. Paths are repository-relative; set `NVU_PROJECT_ROOT` if running from another working directory.


In [ ]:
# Repository-relative paths
PROJECT_ROOT <- Sys.getenv("NVU_PROJECT_ROOT", unset = normalizePath(file.path(getwd(), ".."), mustWork = FALSE))
DATA_DIR <- file.path(PROJECT_ROOT, "data")
RESULTS_DIR <- file.path(PROJECT_ROOT, "results")
REFERENCE_DIR <- file.path(PROJECT_ROOT, "references")
FIGURE_DIR <- file.path(RESULTS_DIR, "figure6")
dir.create(FIGURE_DIR, recursive = TRUE, showWarnings = FALSE)


In [ ]:
library(Seurat)
library(dplyr)
library(tidyr)
library(purrr)
library(FNN)
library(Matrix)
library(ggplot2)
set.seed(112233)
setwd('../results/figure6')

In [ ]:
NVU = readRDS('../results/05_nvu_analysis_results/Hip/200/Hip_merged-subtype_0330.rds')

In [ ]:
NVU

### NVU unit 聚合 + 三组分类 NVU unit 聚合 + 三组分类 

In [ ]:
# ════════════════════════════════════════════════════════════
# 0. 参数
# ════════════════════════════════════════════════════════════

AD_samples <- c("D03556C4", "D01574C4", "C01834C3", "D04303A6",
                "D03556E4", "D01574B6", "D03556D4", "D04303D1",
                "D03556E6", "C01840B1",'AD_1','AD_2')#

Con_samples <- unique(c("D01574A6", "D01574B1", #"D01574C2", "D01574C6",
                         "B03421A5", "B03421A6", "D03556D6", "D03556E2",
                         "D04305A6", "D04305A4", "C04595E2", "C04595F1",
                         "D03556F6", "D03556F4",'control_1','control_2'))#
group_order <- c("Con-Control","AD-Control","AD-Abeta")
target_regions   <- c("CA1","SLRM","FAS")
coord_to_um      <- 0.5
max_um           <- 100
max_coord        <- max_um / coord_to_um        # 200坐标单位
control_min_dist <- 500                          # 坐标单位
boundary_grid_px <- 3

abeta_filled_path <- function(s)
  paste0("../data/abeta/", s, "_abeta_filled.csv")

# ════════════════════════════════════════════════════════════
# 1. 加载斑块数据 & 构建质心/边界索引
# ════════════════════════════════════════════════════════════

message("=== 加载斑块数据 ===")

all_abeta_filled <- lapply(AD_samples, function(s) {
  f <- abeta_filled_path(s)
  if (!file.exists(f)) { warning("缺文件: ", f); return(NULL) }
  df <- read.csv(f)
  if (!"abeta_id" %in% names(df) && "plaque_id" %in% names(df))
    df <- rename(df, abeta_id = plaque_id)
  df %>%
    mutate(sample    = s,
           plaque_id = paste0(s, "_p", abeta_id)) %>%
    select(sample, plaque_id, abeta_id, sp_x, sp_y, n_pixels)
}) %>% bind_rows()

message("斑块总数: ", n_distinct(all_abeta_filled$plaque_id))

# 质心
all_centroids <- all_abeta_filled %>%
  group_by(sample, plaque_id) %>%
  summarise(cx = mean(sp_x), cy = mean(sp_y), .groups = "drop")

# 边界稀疏化索引
thin_boundary <- function(df, gpx = boundary_grid_px) {
  df %>%
    mutate(gx = round(sp_x / gpx), gy = round(sp_y / gpx)) %>%
    distinct(gx, gy, .keep_all = TRUE) %>%
    select(-gx, -gy)
}

abeta_boundary_list <- all_abeta_filled %>%
  group_by(plaque_id) %>% group_split() %>%
  lapply(thin_boundary) %>%
  setNames(sapply(., function(d) d$plaque_id[1]))

message("边界索引构建完成")

# ════════════════════════════════════════════════════════════
# 2. 提取 metadata & 有效 unit
# ════════════════════════════════════════════════════════════

message("\n=== 提取 metadata ===")

meta <- NVU@meta.data %>%
  as.data.frame() %>%
  mutate(cellid = rownames(.))

message("总细胞数: ", nrow(meta))

valid_meta <- meta %>%
  filter(
    !is.na(unit_id), unit_id != "",
    area_m %in% target_regions
  )

message("有效 unit 细胞数: ", nrow(valid_meta))
message("有效 unit 数: ",     n_distinct(valid_meta$unit_id))

                  
                  


In [ ]:
# ════════════════════════════════════════════════════════════
# 重新构建：unit_id 加样本前缀，Con 直接取全部
# ════════════════════════════════════════════════════════════
library(DESeq2)
max_coord_nvu <- 600
ctrl_min_nvu  <- 1000

# ── 1. 重建 valid_meta（加样本前缀）─────────────────────────
valid_meta <- meta %>%
  filter(
    sample_id %in% c(AD_samples, Con_samples),
    !is.na(unit_id), unit_id != "",
    area_m %in% target_regions
  ) %>%
  mutate(unit_id_orig = unit_id,
         unit_id      = paste0(sample_id, "_", unit_id))

message("valid_meta 细胞数: ", nrow(valid_meta))
message("唯一 unit_id 数:   ", n_distinct(valid_meta$unit_id))

# ── 2. Con-Control：直接取所有 Con 样本的 unit ───────────────
con_unit_grouped <- valid_meta %>%
  filter(sample_id %in% Con_samples) %>%
  distinct(unit_id, sample_id, area_m) %>%
  mutate(unit_group     = "Con-Control",
         dist_to_plaque = NA_real_,
         nearest_plaque = NA_character_)

message("Con-Control unit数: ", nrow(con_unit_grouped))
print(table(con_unit_grouped$area_m, con_unit_grouped$sample_id))

# ── 3. AD units：重新计算距离（unit_id已加前缀，坐标不变）────
unit_coords <- valid_meta %>%
  filter(sample_id %in% AD_samples) %>%
  group_by(sample_id, unit_id, area_m) %>%
  summarise(
    ux = mean(x, na.rm = TRUE),
    uy = mean(y, na.rm = TRUE),
    .groups = "drop"
  )

assign_unit_group_sample <- function(unit_df, s) {
  centers_s <- all_centroids %>% filter(sample == s)
  if (nrow(centers_s) == 0) {
    return(unit_df %>%
             mutate(unit_group     = "AD-Control",
                    dist_to_plaque = Inf,
                    nearest_plaque = NA_character_))
  }
  nn_c <- FNN::get.knnx(
    data  = as.matrix(centers_s[, c("cx","cy")]),
    query = as.matrix(unit_df[,   c("ux","uy")]), k = 1
  )
  unit_df$nearest_plaque <- centers_s$plaque_id[nn_c$nn.index[, 1]]

  result <- split(unit_df, unit_df$nearest_plaque) %>%
    imap_dfr(function(udf, pid) {
      bpts <- abeta_boundary_list[[pid]]
      if (is.null(bpts) || nrow(bpts) == 0) {
        udf$dist_to_plaque <- Inf; return(udf)
      }
      nn_f <- FNN::get.knnx(
        data  = as.matrix(bpts[, c("sp_x","sp_y")]),
        query = as.matrix(udf[,  c("ux",   "uy")]), k = 1
      )
      udf$dist_to_plaque <- nn_f$nn.dist[, 1]
      udf
    })

  result %>%
    mutate(unit_group = case_when(
      dist_to_plaque <= max_coord_nvu ~ "AD-Abeta",
      dist_to_plaque >  ctrl_min_nvu  ~ "AD-Control",
      TRUE                            ~ NA_character_
    ))
}

ad_unit_grouped <- unit_coords %>%
  group_by(sample_id) %>%
  group_modify(function(df, key) {
    message("  AD: ", key$sample_id)
    assign_unit_group_sample(df, key$sample_id)
  }) %>%
  ungroup() %>%
  filter(!is.na(unit_group))

message("\nAD-Abeta:   ", sum(ad_unit_grouped$unit_group == "AD-Abeta"))
message("AD-Control: ", sum(ad_unit_grouped$unit_group == "AD-Control"))
print(table(ad_unit_grouped$unit_group, ad_unit_grouped$area_m))

# ── 4. 合并三组 ───────────────────────────────────────────────
all_unit_groups <- bind_rows(
  ad_unit_grouped %>%
    select(unit_id, sample_id, area_m, unit_group, dist_to_plaque),
  con_unit_grouped %>%
    select(unit_id, sample_id, area_m, unit_group, dist_to_plaque)
)

message("\n=== 三组 unit 最终分布 ===")
print(table(all_unit_groups$unit_group, all_unit_groups$area_m))

### Deseq2 ✅️

In [ ]:
# ════════════════════════════════════════════════════════════
# 方案2: 极少量 AD-Control (每样本每区域最多3个) + 稳健 design
# ════════════════════════════════════════════════════════════

set.seed(42)
max_adctrl_per_sample <- 3  # 每样本每区域只保留3个，最大程度稀释其影响

# 排除特定样本的 CA1
all_unit_groups <- all_unit_groups %>%
  dplyr::filter(!(sample_id %in% c("AD_1","AD_2") & area_m == "CA1"))

message("=== 限制前 AD-Control 分布 ===")
all_unit_groups %>%
  dplyr::filter(unit_group == "AD-Control") %>%
  dplyr::count(sample_id, area_m) %>%
  tidyr::pivot_wider(names_from = area_m,
                     values_from = n, values_fill = 0) %>%
  as.data.frame() %>% print()

# AD-Control 下采样
adctrl_downsampled <- all_unit_groups %>%
  dplyr::filter(unit_group == "AD-Control") %>%
  dplyr::group_by(sample_id, area_m) %>%
  dplyr::group_modify(function(df, key) {
    n_take <- min(nrow(df), max_adctrl_per_sample)
    df[sample(nrow(df), n_take, replace = FALSE), ]
  }) %>%
  dplyr::ungroup()

message("\n=== 限制后 AD-Control 分布 ===")
adctrl_downsampled %>%
  dplyr::count(sample_id, area_m) %>%
  tidyr::pivot_wider(names_from = area_m,
                     values_from = n, values_fill = 0) %>%
  as.data.frame() %>% print()

# 重建 all_unit_groups_balanced
all_unit_groups_balanced <- dplyr::bind_rows(
  all_unit_groups %>% dplyr::filter(unit_group == "AD-Abeta"),
  adctrl_downsampled,
  all_unit_groups %>% dplyr::filter(unit_group == "Con-Control")
)

message("\n=== 均衡后三组分布 ===")
print(table(all_unit_groups_balanced$unit_group,
            all_unit_groups_balanced$area_m))

# ════════════════════════════════════════════════════════════
# 重新聚合 pseudo-bulk（用均衡后的 unit）
# ════════════════════════════════════════════════════════════

abeta_unit_ids  <- all_unit_groups_balanced %>%
  dplyr::filter(unit_group == "AD-Abeta")   %>% pull(unit_id)
adctrl_unit_ids <- all_unit_groups_balanced %>%
  dplyr::filter(unit_group == "AD-Control") %>% pull(unit_id)
con_unit_ids    <- all_unit_groups_balanced %>%
  dplyr::filter(unit_group == "Con-Control") %>% pull(unit_id)

cells_abeta  <- valid_meta %>%
  dplyr::filter(unit_id %in% abeta_unit_ids) %>%
  dplyr::mutate(pb_group = "AD-Abeta")
cells_adctrl <- valid_meta %>%
  dplyr::filter(unit_id %in% adctrl_unit_ids) %>%
  dplyr::mutate(pb_group = "AD-Control")
cells_con    <- valid_meta %>%
  dplyr::filter(unit_id %in% con_unit_ids) %>%
  dplyr::mutate(pb_group = "Con-Control")

cells_bulk <- dplyr::bind_rows(cells_abeta, cells_adctrl, cells_con)

message("\n纳入细胞数: ", nrow(cells_bulk))
print(table(cells_bulk$pb_group, cells_bulk$area_m))

# counts 聚合
counts_mat   <- GetAssayData(NVU, assay = "RNA", slot = "counts")
unit_ids_vec <- cells_bulk$unit_id
unique_units <- unique(unit_ids_vec)
message("\n聚合 ", length(unique_units), " 个 unit...")

cell2unit <- Matrix::sparseMatrix(
  i        = match(unit_ids_vec, unique_units),
  j        = seq_along(unit_ids_vec),
  x        = 1,
  dims     = c(length(unique_units), length(unit_ids_vec)),
  dimnames = list(unique_units, cells_bulk$cellid)
)
pb_unit_counts <- t(cell2unit %*% t(counts_mat[, cells_bulk$cellid]))
message("pseudo-bulk: ", nrow(pb_unit_counts), " × ", ncol(pb_unit_counts))

# unit metadata
unit_meta_deseq <- cells_bulk %>%
  dplyr::select(unit_id, sample_id, area_m, pb_group) %>%
  as.data.frame() %>%
  .[!duplicated(.$unit_id), ] %>%
  dplyr::filter(unit_id %in% colnames(pb_unit_counts)) %>%
  dplyr::mutate(
    pb_group  = factor(pb_group,
                       levels = c("Con-Control","AD-Control","AD-Abeta")),
    sample_id = factor(sample_id),
    area_m    = factor(area_m, levels = target_regions)
  ) %>%
  `rownames<-`(NULL) %>%
  tibble::column_to_rownames("unit_id")

pb_unit_counts <- pb_unit_counts[, rownames(unit_meta_deseq)]

message("\n最终 unit 分布（均衡后）:")
print(table(unit_meta_deseq$pb_group, unit_meta_deseq$area_m))

# ════════════════════════════════════════════════════════════
# 构建二分组 metadata
# ════════════════════════════════════════════════════════════

AD_samples <- levels(unit_meta_deseq$sample_id)[
  grepl("^AD", levels(unit_meta_deseq$sample_id))
]

unit_meta_binary <- unit_meta_deseq %>%
  as.data.frame() %>%
  tibble::rownames_to_column("unit_id") %>%
  dplyr::mutate(
    pb_group_binary = factor(
      ifelse(as.character(pb_group) == "AD-Abeta",
             "AD-Abeta", "Non-Abeta"),
      levels = c("Non-Abeta","AD-Abeta")
    ),
    sample_origin = factor(
      ifelse(as.character(sample_id) %in% AD_samples,
             "AD_sample", "Con_sample")
    )
  ) %>%
  tibble::column_to_rownames("unit_id")

message("二分组分布:")
print(table(unit_meta_binary$pb_group_binary, unit_meta_binary$area_m))


# ════════════════════════════════════════════════════════════
# DESeq2 并行设置
# ════════════════════════════════════════════════════════════

library(BiocParallel)
library(DESeq2)

n_cores <- min(6, parallel::detectCores() - 1)
register(MulticoreParam(workers = n_cores))
message("使用 ", n_cores, " 核并行")

# ════════════════════════════════════════════════════════════
# 稳健版 DESeq2 函数
# ════════════════════════════════════════════════════════════

run_deseq2_final <- function(region) {
  message("\n  脑区: ", region)

  keep_units <- rownames(unit_meta_binary)[unit_meta_binary$area_m == region]
  cnt  <- pb_unit_counts[, keep_units, drop = FALSE]
  meta <- unit_meta_binary[keep_units, , drop = FALSE]

  grp_table <- table(meta$pb_group_binary)
  message("  AD-Abeta=", grp_table["AD-Abeta"],
          " Non-Abeta=", grp_table["Non-Abeta"])

  if (any(is.na(grp_table)) || any(grp_table < 2)) {
    message("  -> 某组 unit 数不足，跳过"); return(NULL)
  }

  # ── 固定使用最简 design，不加任何样本协变量 ──────────────────
  # 理由：sample_id / sample_origin 均与 pb_group_binary 共线
  #       (Con样本unit全在Non-Abeta，AD样本unit主要在AD-Abeta)
  #       加协变量会吸收主效应信号，导致0差异基因
  message("  design: ~ pb_group_binary")

  tryCatch({
    cnt_int <- round(as.matrix(cnt))
    storage.mode(cnt_int) <- "integer"

    # 清理 sample_id factor levels，避免字符警告
    meta$sample_id <- droplevels(meta$sample_id)

    dds <- DESeqDataSetFromMatrix(
      countData = cnt_int,
      colData   = meta,
      design    = ~ pb_group_binary
    )

    # 基因过滤：两组各自至少3个 unit 有表达
    group_vec  <- meta$pb_group_binary
    keep_abeta <- rowSums(cnt_int[, group_vec == "AD-Abeta",  drop = FALSE] > 0) >= 3
    keep_nonab <- rowSums(cnt_int[, group_vec == "Non-Abeta", drop = FALSE] > 0) >= 3
    dds <- dds[keep_abeta | keep_nonab, ]
    message("  过滤后基因数: ", nrow(dds))

    dds <- estimateSizeFactors(dds, type = "poscounts")
    dds <- DESeq(
      dds,
      fitType  = "local",   # 数据量大时 local 比 parametric 更稳健
      parallel = FALSE,     # 关掉并行，避免 zmq 中断错误
      quiet    = TRUE
    )

    res <- results(
      dds,
      contrast    = c("pb_group_binary", "AD-Abeta", "Non-Abeta"),
      alpha       = 0.05,
      cooksCutoff = FALSE
    ) %>%
      as.data.frame() %>%
      tibble::rownames_to_column("gene") %>%
      dplyr::filter(!is.na(padj)) %>%
      dplyr::mutate(
        area_m    = region,
        direction = dplyr::case_when(
          padj < 0.05 & log2FoldChange >  0 ~ "up",
          padj < 0.05 & log2FoldChange <  0 ~ "down",
          TRUE ~ "ns"
        )
      ) %>%
      dplyr::arrange(padj, dplyr::desc(abs(log2FoldChange)))

    message("  上调: ", sum(res$direction == "up"),
            "  下调: ", sum(res$direction == "down"))
    res

  }, error = function(e) {
    message("  -> 错误: ", conditionMessage(e)); NULL
  })
}
# ════════════════════════════════════════════════════════════
# 运行 DESeq2
# ════════════════════════════════════════════════════════════

message("\n=== DESeq2 方案2（稳健版）===")
t0 <- proc.time()
deseq_results       <- lapply(target_regions, run_deseq2_final)
names(deseq_results) <- target_regions
message("耗时: ", round((proc.time() - t0)[3], 1), " 秒")

# ════════════════════════════════════════════════════════════
# 整理结果
# ════════════════════════════════════════════════════════════

all_deseq <- dplyr::bind_rows(deseq_results) %>%
  dplyr::filter(!is.na(padj)) %>%
  dplyr::mutate(area_m = factor(area_m, levels = target_regions))

sig_deseq <- all_deseq %>% dplyr::filter(direction != "ns")

message("\n=== 显著基因汇总 ===")
print(table(sig_deseq$area_m, sig_deseq$direction))

write.csv(all_deseq, "NVU_deseq2_binary_all.csv",  row.names = FALSE)
write.csv(sig_deseq, "NVU_deseq2_binary_sig.csv",  row.names = FALSE)

message("完成")

### 差异基因分析

In [ ]:
# ════════════════════════════════════════════════════════════
# 火山图（修复版）
# ════════════════════════════════════════════════════════════

library(ggplot2)
library(ggrepel)
library(dplyr)

all_deseq  <- read.csv('../results/figure6/NVU_deseq2_binary_all.csv')
sig_deseq  <- read.csv('../results/figure6/NVU_deseq2_binary_sig.csv')

target_regions <- c("CA1", "SLRM", "FAS")
region_colors  <- c("CA1" = "#C0322F", "SLRM" = "#26AAE7", "FAS" = "#51AA44")

all_deseq <- all_deseq %>%
  dplyr::mutate(area_m = factor(area_m, levels = target_regions))

# ── 每个区域取 top10：优先按 padj，再按 |FC| ──────────────────
top_genes <- lapply(target_regions, function(r) {
  sig_deseq %>%
    dplyr::filter(area_m == r) %>%
    dplyr::distinct(gene, .keep_all = TRUE) %>%
    dplyr::arrange(padj, dplyr::desc(abs(log2FoldChange))) %>%
    dplyr::slice_head(n = 10)
}) %>%
  dplyr::bind_rows() %>%
  dplyr::mutate(area_m = factor(area_m, levels = target_regions))

# ── 给 all_deseq 加颜色分组列（用于底层点着色）────────────────
all_deseq <- all_deseq %>%
  dplyr::mutate(
    point_color = dplyr::case_when(
      padj < 0.05 & log2FoldChange >  0.75 ~ "up",
      padj < 0.05 & log2FoldChange <  -0.75 ~ "down",
      TRUE ~ "ns"
    )
  )

p_volcano <- ggplot() +
  geom_point(
    data  = all_deseq %>% dplyr::filter(point_color == "ns"),
    aes(x = log2FoldChange, y = -log10(padj)),
    size = 0.6, color = "grey82", alpha = 0.5
  ) +
  geom_point(
    data  = all_deseq %>% dplyr::filter(point_color != "ns"),
    aes(x = log2FoldChange, y = -log10(padj), color = area_m),
    size = 0.9, alpha = 0.7
  ) +
  geom_point(
    data  = top_genes,
    aes(x = log2FoldChange, y = -log10(padj), color = area_m),
    size  = 2.2, alpha = 1, shape = 16
  ) +
  geom_hline(yintercept = -log10(0.05),
             linewidth = 0.4, color = "grey50", linetype = "dashed") +
  geom_vline(xintercept = 0,
             linewidth = 0.4, color = "grey50", linetype = "dashed") +
  ggrepel::geom_text_repel(
    data               = top_genes,
    aes(x = log2FoldChange, y = -log10(padj),
        label = gene, color = area_m),
    size               = 2.6,
    fontface           = "bold.italic",
    max.overlaps       = 50,
    segment.size       = 0.3,
    segment.color      = "grey50",
    box.padding        = 0.4,
    point.padding      = 0.2,
    force              = 3,
    show.legend        = FALSE,
    min.segment.length = 0.1,
    direction          = "both"   # 竖排时允许上下左右都能推开
  ) +
  # ── 竖向排列，去掉 coord_flip ──────────────────────────────
  facet_grid(area_m ~ ., scales = "free_y") +   # 竖排，Y轴各自独立
  scale_color_manual(values = region_colors) +
  labs(
    title    = expression("NVU DESeq2 — AD-A"*beta*" vs Non-Abeta"),
    subtitle = paste0(
      "Up: ",   sum(sig_deseq$direction == "up"),
      "  Down: ", sum(sig_deseq$direction == "down"),
      "  |  design: ~ pb_group_binary"
    ),
    x     = expression(log[2]*"FC (AD-A"*beta*" / Non-Abeta)"),
    y     = expression(-log[10]*"(adj. p)"),
    color = NULL
  ) +
  theme_bw(base_size = 11) +
  theme(
    legend.position  = "none",
    panel.grid       = element_blank(),
    axis.text        = element_text(size = 9, color = "black"),
    strip.text.y     = element_text(size = 11, face = "bold"),  # 竖排用 strip.text.y
    strip.background = element_rect(fill = "grey95", color = NA),
    plot.title       = element_text(face = "bold", size = 12, hjust = 0.5),
    plot.subtitle    = element_text(size = 9, color = "grey40", hjust = 0.5),
    panel.border     = element_rect(color = "grey70", linewidth = 0.5),
    plot.margin      = margin(10, 10, 10, 10)
  )

cairo_pdf("../results/figure6/NVU_deseq2_volcano_vertical.pdf",
          width = 8, height = 12)   # 竖排改成高图
print(p_volcano)
dev.off()

ggsave("../results/figure6/NVU_deseq2_volcano_vertical.png",
       p_volcano, width = 8, height = 12, dpi = 300)

message("完成")



In [ ]:
Genes <- read.csv('../results/figure6/NVU_deseq2_binary_sig.csv')
Genes = Genes[abs(Genes$log2FoldChange)>0.75,]
table(Genes$area_m,Genes$direction)

library(UpSetR)
library(dplyr)

# ════════════════════════════════════════════════════════════
# 1. 提取各脑区上调基因
# ════════════════════════════════════════════════════════════

up_genes <- Genes %>%
  dplyr::filter(direction == "up")

gene_CA1  <- up_genes %>% dplyr::filter(area_m == "CA1")  %>% dplyr::pull(gene)
gene_SLRM <- up_genes %>% dplyr::filter(area_m == "SLRM") %>% dplyr::pull(gene)
gene_FAS  <- up_genes %>% dplyr::filter(area_m == "FAS")  %>% dplyr::pull(gene)

message("CA1:  ", length(gene_CA1),  " genes")
message("SLRM: ", length(gene_SLRM), " genes")
message("FAS:  ", length(gene_FAS),  " genes")
message("Shared CA1∩SLRM∩FAS: ",
        length(Reduce(intersect, list(gene_CA1, gene_SLRM, gene_FAS))))

# ════════════════════════════════════════════════════════════
# 2. 构建 UpSetR 输入格式（membership list）
# ════════════════════════════════════════════════════════════

gene_list <- list(
  CA1  = gene_CA1,
  SLRM = gene_SLRM,
  FAS  = gene_FAS
)

# fromList 转为 binary membership matrix
upset_input <- UpSetR::fromList(gene_list)

message("\n数据矩阵维度: ", nrow(upset_input), " × ", ncol(upset_input))

# ════════════════════════════════════════════════════════════
# 3. UpSet 图
# ════════════════════════════════════════════════════════════

region_colors <- c(CA1  = "#3C5488",
                   SLRM = "#00A087",
                   FAS  = "#7E6148")

library(UpSetR)
library(grid)

draw_upset <- function() {
  # 必须用 print() 才能在非交互设备上渲染
  p <- upset(
    upset_input,
    sets           = c("CA1","SLRM","FAS"),
    sets.bar.color = c("#3C5488","#00A087","#7E6148"),
    main.bar.color = "#2C3E50",
    matrix.color   = "#2C3E50",
    point.size     = 3.5,
    line.size      = 0.8,
    order.by       = "freq",
    decreasing     = TRUE,
    show.numbers   = "yes",
    number.angles  = 0,
    text.scale     = c(1.4, 1.2, 1.2, 1.1, 1.3, 1.1),
    mainbar.y.label = "Intersection size",
    sets.x.label    = "Set size",
    mb.ratio        = c(0.65, 0.35),
    queries = list(
      list(
        query      = intersects,
        params     = list("CA1","SLRM","FAS"),
        color      = "#E64B35",
        active     = TRUE,
        query.name = "Shared by all"
      )
    ),
    query.legend = "bottom"
  )
  print(p)   # ← 关键

  grid::grid.text(
    label = expression(bold("A"*beta*"-associated up-regulated genes")),
    x     = 0.65,
    y     = 0.98,
    gp    = grid::gpar(fontsize = 14, fontface = "bold")
  )
}

# PDF
cairo_pdf("Abeta_associated_genes_upset.pdf", width = 9, height = 6)
draw_upset()
dev.off()

# PNG
png("Abeta_associated_genes_upset.png",
    width = 9, height = 6, units = "in", res = 300)
draw_upset()
dev.off()

message("完成")
# PDF
cairo_pdf("Abeta_associated_genes_upset.pdf", width = 9, height = 6)
draw_upset()
dev.off()

# PNG
png("Abeta_associated_genes_upset.png",
    width = 9, height = 6, units = "in", res = 300)
draw_upset()
dev.off()

message("完成: Abeta_associated_genes_upset.pdf/.png")

# ════════════════════════════════════════════════════════════
# 4. 输出各交集的基因列表
# ════════════════════════════════════════════════════════════

# 各交集基因
shared_all   <- Reduce(intersect, list(gene_CA1, gene_SLRM, gene_FAS))
shared_CA1_SLRM <- setdiff(intersect(gene_CA1, gene_SLRM), gene_FAS)
shared_CA1_FAS  <- setdiff(intersect(gene_CA1, gene_FAS),  gene_SLRM)
shared_SLRM_FAS <- setdiff(intersect(gene_SLRM, gene_FAS), gene_CA1)
unique_CA1   <- setdiff(gene_CA1,  c(gene_SLRM, gene_FAS))
unique_SLRM  <- setdiff(gene_SLRM, c(gene_CA1,  gene_FAS))
unique_FAS   <- setdiff(gene_FAS,  c(gene_CA1,  gene_SLRM))

message("\n=== 各交集基因数 ===")
message("Shared CA1∩SLRM∩FAS: ", length(shared_all))
message("Shared CA1∩SLRM:     ", length(shared_CA1_SLRM))
message("Shared CA1∩FAS:      ", length(shared_CA1_FAS))
message("Shared SLRM∩FAS:     ", length(shared_SLRM_FAS))
message("Unique CA1:          ", length(unique_CA1))
message("Unique SLRM:         ", length(unique_SLRM))
message("Unique FAS:          ", length(unique_FAS))

message("\nShared by all regions:")
print(shared_all)

# 保存交集表
intersection_df <- bind_rows(
  tibble(gene = shared_all,       intersection = "CA1_SLRM_FAS"),
  tibble(gene = shared_CA1_SLRM,  intersection = "CA1_SLRM"),
  tibble(gene = shared_CA1_FAS,   intersection = "CA1_FAS"),
  tibble(gene = shared_SLRM_FAS,  intersection = "SLRM_FAS"),
  tibble(gene = unique_CA1,       intersection = "CA1_only"),
  tibble(gene = unique_SLRM,      intersection = "SLRM_only"),
  tibble(gene = unique_FAS,       intersection = "FAS_only")
)

write.csv(intersection_df,
          "Abeta_associated_genes_intersections.csv",
          row.names = FALSE)

message("\n完成:")
message("  Abeta_associated_genes_upset.pdf/png")
message("  Abeta_associated_genes_intersections.csv")

In [ ]:
# ════════════════════════════════════════════════════════════
# 富集分析：29个共有基因 + 各脑区特异基因
# ════════════════════════════════════════════════════════════

library(clusterProfiler)
library(org.Hs.eg.db)
library(ggplot2)
library(dplyr)
library(patchwork)
library(stringr)

# ════════════════════════════════════════════════════════════
# 1. 基因分组
# ════════════════════════════════════════════════════════════

shared_all  <- Reduce(intersect, list(gene_CA1, gene_SLRM, gene_FAS))
unique_CA1  <- setdiff(gene_CA1,  c(gene_SLRM, gene_FAS))
unique_SLRM <- setdiff(gene_SLRM, c(gene_CA1,  gene_FAS))
unique_FAS  <- setdiff(gene_FAS,  c(gene_CA1,  gene_SLRM))

message("Shared (all 3): ", length(shared_all))
message("Unique CA1:     ", length(unique_CA1))
message("Unique SLRM:    ", length(unique_SLRM))
message("Unique FAS:     ", length(unique_FAS))
print(shared_all)

gene_sets <- list(
  "Shared\n(CA1+SLRM+FAS)" = shared_all,
  "CA1 specific"           = unique_CA1,
  "SLRM specific"          = unique_SLRM,
  "FAS specific"           = unique_FAS
)

# ════════════════════════════════════════════════════════════
# 2. SYMBOL → Entrez 转换
# ════════════════════════════════════════════════════════════

entrez_sets <- lapply(names(gene_sets), function(nm) {
  genes <- gene_sets[[nm]]
  message("转换 [", nm, "]: ", length(genes), " genes")
  if (length(genes) < 3) return(NULL)
  df <- bitr(genes,
             fromType = "SYMBOL",
             toType   = "ENTREZID",
             OrgDb    = org.Hs.eg.db)
  message("  成功: ", nrow(df))
  df$ENTREZID
})
names(entrez_sets) <- names(gene_sets)

# 过滤空集
entrez_sets <- entrez_sets[!sapply(entrez_sets, is.null)]

# ════════════════════════════════════════════════════════════
# 3. compareCluster GO BP（四组并排）
# ════════════════════════════════════════════════════════════

message("\n=== compareCluster GO BP ===")

compare_go <- tryCatch({
  compareCluster(
    geneClusters  = entrez_sets,
    fun           = "enrichGO",
    OrgDb         = org.Hs.eg.db,
    ont           = "BP",
    pAdjustMethod = "BH",
    pvalueCutoff  = 0.05,
    qvalueCutoff  = 0.2,
    readable      = TRUE
  )
}, error = function(e) {
  message("compareCluster 失败: ", conditionMessage(e)); NULL
})

# ════════════════════════════════════════════════════════════
# 重新跑富集（SLRM 放宽阈值）+ 手动筛选通路 + 合并图
# ════════════════════════════════════════════════════════════

run_go_bp_flexible <- function(gene_ids, label,
                                pval = 0.05, qval = 0.2) {
  if (is.null(gene_ids) || length(gene_ids) < 3) return(NULL)
  message("GO BP [", label, "] pval=", pval, ": ", length(gene_ids), " genes")
  tryCatch({
    enrichGO(
      gene          = gene_ids,
      OrgDb         = org.Hs.eg.db,
      ont           = "BP",
      pAdjustMethod = "BH",
      pvalueCutoff  = pval,
      qvalueCutoff  = qval,
      readable      = TRUE
    )
  }, error = function(e) { message("失败: ", conditionMessage(e)); NULL })
}




In [ ]:
library(dplyr)
library(purrr)

# SLRM 放宽到 p<0.2，其余保持 p<0.05
go_list <- list(
  "Shared\n(CA1+SLRM+FAS)" = run_go_bp_flexible(
    entrez_sets[["Shared\n(CA1+SLRM+FAS)"]], "Shared", pval = 0.05),
  "CA1 specific"  = run_go_bp_flexible(
    entrez_sets[["CA1 specific"]],  "CA1",  pval = 0.05),
  "SLRM specific" = run_go_bp_flexible(
    entrez_sets[["SLRM specific"]], "SLRM", pval = 0.05),
  "FAS specific"  = run_go_bp_flexible(
    entrez_sets[["FAS specific"]],  "FAS",  pval = 0.05)
)


go_table <- imap_dfr(go_list, function(obj, name) {
  as.data.frame(obj) %>%
    mutate(gene_set = name, .before = 1)
})

write.csv(go_table, "GO_enrichment_all.csv", row.names = FALSE)

message("完成: ", nrow(go_table), " 行，",
        "覆盖 ", n_distinct(go_table$gene_set), " 个基因集")
print(table(go_table$gene_set))

In [ ]:
# ════════════════════════════════════════════════════════════
# 手动筛选通路 + 合并点图
# ════════════════════════════════════════════════════════════

selected_terms <- list(
  "Shared\n(CA1+SLRM+FAS)" = c(
    "GO:1901216",  # positive regulation of neuron death
    "GO:1990000",  # amyloid fibril formation
    "GO:0070997",  # neuron death
    "GO:0006457",  # protein folding
    "GO:0042026",  # protein refolding
    "GO:0010506",  # regulation of autophagy
    "GO:0010001",  # glial cell differentiation
    "GO:0021782"   # glial cell development
  ),
  "CA1 specific" = c(
    "GO:0035767",  # endothelial cell chemotaxis
    "GO:0007162",  # negative regulation of cell adhesion
    "GO:0045766" # cell-substrate adhesion

  ),
  "SLRM specific" = c(

    "GO:0097193",  # cell-matrix adhesion
    "GO:0031589",  # cell-substrate adhesion
    "GO:0018108",  # peptidyl-tyrosine phosphorylation
    "GO:1903037"   # regulation of leukocyte cell-cell adhesion
  ),
  "FAS specific" = c(
    "GO:0022900"  # electron transport chain


  )
)


selected_table <- imap_dfr(selected_terms, function(go_ids, set_name) {

    
  matched_key <- names(go_list)[
    gsub("\n", "\n", names(go_list)) == set_name
  ]
  
  if (length(matched_key) == 0) {
    message("未找到 gene_set: ", set_name)
    return(NULL)
  }
  
  as.data.frame(go_list[[matched_key]]) %>%
    filter(ID %in% go_ids) %>%
    mutate(gene_set = set_name, .before = 1)
}) %>%
  mutate(
    gene_set = factor(gene_set, levels = names(selected_terms)),
    ID       = factor(ID, levels = unique(unlist(selected_terms)))  # ← 加 unique()
  ) %>%
  arrange(gene_set, ID) %>%
  mutate(gene_set = as.character(gene_set),
         ID       = as.character(ID))


# ── 保存 ────────────────────────────────────────────────────
write.csv(selected_table,
          "GO_selected_terms.csv",
          row.names = FALSE)

message("完成: ", nrow(selected_table), " 行")
print(table(selected_table$gene_set))

# ════════════════════════════════════════════════════════════
# 构建画图数据
# ════════════════════════════════════════════════════════════

get_plot_df <- function(res, nm) {
  if (is.null(res) || nrow(res@result) == 0) return(NULL)
  terms <- selected_terms[[nm]]
  df <- res@result %>%
    dplyr::filter(ID %in% terms) %>%
    dplyr::mutate(
      Description   = stringr::str_wrap(Description, width = 42),
      GeneRatio_num = sapply(GeneRatio, function(x) {
        v <- as.numeric(strsplit(x, "/")[[1]]); v[1]/v[2]
      }),
      gene_set = nm
    )
  if (nrow(df) == 0) return(NULL)
  df
}

plot_dfs <- lapply(names(go_list), function(nm) {
  get_plot_df(go_list[[nm]], nm)
})
names(plot_dfs) <- names(go_list)

all_df <- dplyr::bind_rows(plot_dfs) %>%
  dplyr::filter(!is.na(Description)) %>%
  dplyr::mutate(
    gene_set = factor(gene_set, levels = names(selected_terms))
  )

message("各组纳入通路数:")
print(table(all_df$gene_set))

# ════════════════════════════════════════════════════════════
# 合并点图
# ════════════════════════════════════════════════════════════

set_colors <- c(
  "Shared\n(CA1+SLRM+FAS)" = "#E64B35",
  "CA1 specific"            = "#3C5488",
  "SLRM specific"           = "#00A087",
  "FAS specific"            = "#7E6148"
)

# 每个分面内按 GeneRatio 独立排序
all_df <- all_df %>%
  dplyr::group_by(gene_set) %>%
  dplyr::arrange(GeneRatio_num, .by_group = TRUE) %>%
  dplyr::mutate(
    desc_ordered = factor(Description,
                           levels = unique(Description))
  ) %>%
  dplyr::ungroup()

p_final <- ggplot(all_df,
                   aes(x    = GeneRatio_num,
                       y    = desc_ordered,
                       size = Count,
                       color = p.adjust)) +
  geom_point(alpha = 0.9) +
  facet_wrap(
    ~ gene_set,
    scales   = "free_y",
    ncol     = 2
  ) +
  scale_color_gradient(
    low  = "#B2182B",
    high = "#DEEBF7",
    name = "adj. p-value"
  ) +
  scale_size_continuous(
    name   = "Gene count",
    range  = c(2, 9),
    breaks = c(2, 5, 8, 12)
  ) +
  scale_x_continuous(expand = expansion(mult = c(0.05, 0.25))) +
  labs(
    title   = expression(bold("GO BP enrichment — A"*beta*"-associated up-regulated genes")),
    caption = "SLRM: p < 0.2 (exploratory); others: BH-adjusted p < 0.05",
    x       = "Gene ratio",
    y       = NULL
  ) +
  theme_classic(base_size = 11) +
  theme(
    plot.title       = element_text(face = "bold", size = 12,
                                     hjust = 0.5,
                                     margin = margin(b = 8)),
    plot.caption     = element_text(size = 8, color = "grey50",
                                     hjust = 0),
    strip.background = element_blank(),
    strip.text       = element_text(face = "bold", size = 11),
    axis.text.y      = element_text(size = 9,  color = "black"),
    axis.text.x      = element_text(size = 9,  color = "black"),
    axis.title.x     = element_text(size = 10, margin = margin(t = 6)),
    axis.line        = element_line(linewidth = 0.4),
    axis.ticks       = element_line(linewidth = 0.35),
    legend.position  = "right",
    legend.text      = element_text(size = 8),
    legend.title     = element_text(size = 9),
    panel.spacing    = unit(1, "cm"),
    plot.margin      = margin(10, 10, 8, 10)
  )

# 动态计算图片高度
n_terms <- all_df %>%
  dplyr::group_by(gene_set) %>%
  dplyr::summarise(n = n_distinct(Description)) %>%
  dplyr::pull(n)
fig_h <- max(10, max(n_terms) * 0.45 * 2 + 3)
fig_h <- min(fig_h, 35)

cairo_pdf("GO_BP_combined.pdf", width = 16, height = fig_h)
print(p_final)
dev.off()

ggsave("GO_BP_combined.png", p_final,
       width = 16, height = fig_h,
       dpi = 300, limitsize = FALSE)

message("完成: GO_BP_combined.pdf/.png  (",
        round(16), " × ", round(fig_h), " in)")

In [ ]:
# ════════════════════════════════════════════════════════════
# 构建画图数据（同前）
# ════════════════════════════════════════════════════════════

get_plot_df <- function(res, nm) {
  if (is.null(res) || nrow(res@result) == 0) return(NULL)
  terms <- selected_terms[[nm]]
  df <- res@result %>%
    dplyr::filter(ID %in% terms) %>%
    dplyr::mutate(
      Description   = stringr::str_wrap(Description, width = 35),
      GeneRatio_num = sapply(GeneRatio, function(x) {
        v <- as.numeric(strsplit(x, "/")[[1]]); v[1]/v[2]
      }),
      gene_set = nm
    )
  if (nrow(df) == 0) return(NULL)
  df
}

plot_dfs <- lapply(names(go_list), function(nm) {
  get_plot_df(go_list[[nm]], nm)
})
names(plot_dfs) <- names(go_list)

all_df <- dplyr::bind_rows(plot_dfs) %>%
  dplyr::filter(!is.na(Description))

# ════════════════════════════════════════════════════════════
# 构建 y 轴顺序：按分组分块，组内按 GeneRatio 升序
# ════════════════════════════════════════════════════════════

# 每个 Description 所属的 gene_set（用第一次出现的）
desc_order <- all_df %>%
  dplyr::mutate(
    gene_set = factor(gene_set, levels = names(selected_terms))
  ) %>%
  dplyr::arrange(gene_set, GeneRatio_num) %>%
  # 同一 Description 可能出现在多个 gene_set（如 GO:0031589）
  # 保留每条记录，y 轴 label 加上 gene_set 前缀以区分
  dplyr::mutate(
    desc_label = ifelse(
      duplicated(Description) | duplicated(Description, fromLast = TRUE),
      paste0(Description, "\n[", gsub("\n"," ", gene_set), "]"),
      Description
    )
  )

# y 轴顺序（从下到上：FAS → SLRM → CA1 → Shared）
y_levels <- desc_order %>%
  dplyr::arrange(gene_set, GeneRatio_num) %>%
  dplyr::pull(desc_label) %>%
  unique()

all_df_plot <- desc_order %>%
  dplyr::mutate(
    gene_set   = factor(gene_set,   levels = names(selected_terms)),
    desc_label = factor(desc_label, levels = y_levels)
  )

# ════════════════════════════════════════════════════════════
# 分组颜色（用于 y 轴文字着色）
# ════════════════════════════════════════════════════════════

set_colors <- c(
  "Shared\n(CA1+SLRM+FAS)" = "#E64B35",
  "CA1 specific"            = "#3C5488",
  "SLRM specific"           = "#00A087",
  "FAS specific"            = "#7E6148"
)

# y 轴每条 label 对应的颜色
label_colors <- desc_order %>%
  dplyr::arrange(gene_set, GeneRatio_num) %>%
  dplyr::distinct(desc_label, gene_set) %>%
  dplyr::mutate(color = set_colors[as.character(gene_set)]) %>%
  dplyr::arrange(factor(desc_label, levels = y_levels)) %>%
  dplyr::pull(color)

# ════════════════════════════════════════════════════════════
# 绘图
# ════════════════════════════════════════════════════════════

p_dot <- ggplot(all_df_plot,
                aes(x     = gene_set,
                    y     = desc_label,
                    size  = Count,
                    color = pvalue)) +
  geom_point(alpha = 0.9) +
  # 分组分隔线（在 y 轴上画横向背景带）
  annotate("rect",
           xmin = -Inf, xmax = Inf,
           ymin = 0.5,
           ymax = length(y_levels) + 0.5,
           fill = NA, color = NA) +
  scale_color_gradient(
    low    = "#B2182B",
    high   = "#DEEBF7",
    name   = "adj. p-value",
    limits = c(0, max(all_df_plot$pvalue, na.rm = TRUE)),
    guide  = guide_colorbar(reverse = TRUE)
  ) +
  scale_size_continuous(
    name   = "Gene count",
    range  = c(2, 9),
    breaks = c(2, 5, 8, 12)
  ) +
  scale_x_discrete(
    labels = function(x) gsub("\n", "\n", x),
    position = "bottom"
  ) +
  scale_y_discrete(limits = y_levels) +
  labs(
    title   = expression(bold("GO BP enrichment — A"*beta*"-associated up-regulated genes")),
    caption = "SLRM: p < 0.2 (exploratory); others: BH-adjusted p < 0.05",
    x       = NULL,
    y       = NULL
  ) +
  theme_classic(base_size = 11) +
  theme(
    plot.title       = element_text(face = "bold", size = 12,
                                     hjust = 0.5,
                                     margin = margin(b = 10)),
    plot.caption     = element_text(size = 8, color = "grey50", hjust = 0),
    axis.text.x      = element_text(size = 10, color = "black",
                                     face = "bold", angle = 0, hjust = 0.5),
    axis.text.y      = element_text(size = 9,  color = label_colors),
    axis.line        = element_line(linewidth = 0.4),
    axis.ticks.x     = element_blank(),
    axis.ticks.y     = element_line(linewidth = 0.35),
    legend.position  = "right",
    legend.text      = element_text(size = 8),
    legend.title     = element_text(size = 9),
    panel.grid.major.y = element_line(color = "grey92", linewidth = 0.4),
    panel.grid.major.x = element_line(color = "grey88", linewidth = 0.4),
    plot.margin      = margin(10, 10, 8, 10)
  )

# ── 添加分组背景色带 ──────────────────────────────────────
# 计算每个 gene_set 的 y 范围
group_y <- all_df_plot %>%
  dplyr::distinct(desc_label, gene_set) %>%
  dplyr::mutate(y_pos = as.numeric(factor(desc_label, levels = y_levels))) %>%
  dplyr::group_by(gene_set) %>%
  dplyr::summarise(y_min = min(y_pos) - 0.5,
                   y_max = max(y_pos) + 0.5,
                   .groups = "drop")

band_colors <- c(
  "Shared\n(CA1+SLRM+FAS)" = "#FDECEA",
  "CA1 specific"            = "#EBF0FA",
  "SLRM specific"           = "#E6F5F2",
  "FAS specific"            = "#F5F0EB"
)

for (i in seq_len(nrow(group_y))) {
  gs <- as.character(group_y$gene_set[i])
  p_dot <- p_dot +
    annotate("rect",
             xmin = -Inf, xmax = Inf,
             ymin = group_y$y_min[i],
             ymax = group_y$y_max[i],
             fill  = band_colors[gs],
             alpha = 0.5,
             color = NA)
}

# annotate 在 geom_point 下面，需要重新叠加点
p_dot <- p_dot +
  geom_point(alpha = 0.9)

# ════════════════════════════════════════════════════════════
# 保存
# ════════════════════════════════════════════════════════════

n_rows  <- length(y_levels)
fig_h   <- max(6, n_rows * 0.38 + 2.5)
fig_h   <- min(fig_h, 35)

cairo_pdf("GO_BP_dotplot.pdf", width = 8, height = fig_h)
print(p_dot)
dev.off()

ggsave("GO_BP_dotplot.png", p_dot,
       width = 9, height = fig_h,
       dpi = 300, limitsize = FALSE)

message("完成: GO_BP_dotplot.pdf/.png  (9 × ", round(fig_h, 1), " in)")

In [ ]:
all_df_lol

### 热图

In [ ]:
# ════════════════════════════════════════════════════════════
# 热图：DESeq2 结果直接驱动
# 用 DESeq2 的 up-regulated genes，表达值来自 NVU_unit
# ════════════════════════════════════════════════════════════

library(pheatmap)
library(dplyr)
library(tidyr)
library(tibble)
library(Seurat)

breaks_seq   <- seq(-2.5, 2.5, length.out = 101)
color_pal    <- colorRampPalette(c("#2166AC","#F7F7F7","#B2182B"))(100)
group_order  <- c("Con-Control","AD-Control","AD-Abeta")
region_order <- c("CA1","SLRM","FAS")
n_top        <- 30

# 读取 DESeq2 结果
Genes    <- read.csv('../results/figure6/NVU_deseq2_binary_sig.csv')
NVU_unit <- readRDS('../results/figure6/NVU_unit_pseudobulk.rds')

# 去除 XIST
Genes <- Genes %>% dplyr::filter(gene != "XIST", direction == "up")

message("DESeq2 up genes:")
print(table(Genes$area_m))
message("\nNVU_unit 分布:")
print(table(NVU_unit$unit_group, NVU_unit$area_m))

# ════════════════════════════════════════════════════════════
# 提取表达矩阵 & 计算均值
# ════════════════════════════════════════════════════════════

all_genes_plot <- Genes %>%
  dplyr::filter(area_m %in% region_order) %>%
  dplyr::pull(gene) %>%
  unique() %>%
  .[. %in% rownames(NVU_unit)]

expr_mat <- GetAssayData(NVU_unit, assay = "RNA", slot = "data")
expr_sub  <- as.matrix(expr_mat[all_genes_plot, , drop = FALSE])

unit_meta_df <- NVU_unit@meta.data %>%
  as.data.frame() %>%
  mutate(unit_id = rownames(.)) %>%
  dplyr::filter(unit_group %in% group_order,
                area_m    %in% region_order)

mean_expr_all <- lapply(all_genes_plot, function(g) {
  vals <- expr_sub[g, ]
  unit_meta_df %>%
    mutate(expr = vals[unit_id]) %>%
    dplyr::group_by(unit_group, area_m) %>%
    dplyr::summarise(mean_expr = mean(expr, na.rm = TRUE),
                     .groups = "drop") %>%
    mutate(gene = g)
}) %>% dplyr::bind_rows()

# ════════════════════════════════════════════════════════════
# 预计算 shared 基因集
# ════════════════════════════════════════════════════════════

top_per_region <- lapply(region_order, function(r) {
  Genes %>%
    dplyr::filter(area_m == r) %>%
    dplyr::arrange(desc(log2FoldChange)) %>%
    dplyr::slice_head(n = n_top) %>%
    dplyr::pull(gene)
})
names(top_per_region) <- region_order

shared_genes_set <- Reduce(intersect, top_per_region) %>%
  .[. %in% rownames(NVU_unit)]

if (length(shared_genes_set) == 0) {
  all_top          <- unlist(top_per_region)
  shared_genes_set <- names(table(all_top)[table(all_top) >= 2]) %>%
    .[. %in% rownames(NVU_unit)]
}
message("shared 基因集大小: ", length(shared_genes_set))

# ════════════════════════════════════════════════════════════
# 通用绘图函数
# ════════════════════════════════════════════════════════════

draw_heatmap <- function(mat_z, col_anno, anno_colors,
                         title, file_prefix,
                         gaps_col  = NULL,
                         cellwidth = 35) {

  fig_height <- max(4, nrow(mat_z) * 0.38 + 2.5)
  fig_width  <- max(5, ncol(mat_z) * 0.9  + 3.5)

  args <- list(
    mat_z,
    color             = color_pal,
    breaks            = breaks_seq,
    cluster_rows      = TRUE,
    cluster_cols      = FALSE,
    annotation_col    = col_anno,
    annotation_colors = anno_colors,
    gaps_col          = gaps_col,
    border_color      = "white",
    cellwidth         = cellwidth,
    cellheight        = 16,
    fontsize_row      = 10,
    fontsize_col      = 10,
    angle_col         = 45,
    show_rownames     = TRUE,
    show_colnames     = TRUE,
    legend_breaks     = c(-2.5, -1, 0, 1, 2.5),
    legend_labels     = c("-2.5", "-1", "0", "1", "2.5"),
    main              = title
  )

  do.call(pheatmap, c(args, list(
    filename = paste0(file_prefix, ".pdf"),
    width    = fig_width,
    height   = fig_height
  )))

  p <- do.call(pheatmap, c(args, list(silent = TRUE)))
  png(paste0(file_prefix, ".png"),
      width = fig_width, height = fig_height,
      units = "in", res = 300)
  grid::grid.newpage()
  grid::grid.draw(p$gtable)
  dev.off()

  message("  保存: ", file_prefix, ".pdf/.png")
  invisible(p)
}

# ════════════════════════════════════════════════════════════
# Shared 热图（三脑区均有的上调基因）
# ════════════════════════════════════════════════════════════

make_shared_heatmap <- function() {
  message("\n=== Shared (CA1 + SLRM + FAS) ===")
  if (length(shared_genes_set) == 0) { message("跳过"); return(NULL) }

  mat_r <- mean_expr_all %>%
    dplyr::filter(gene %in% shared_genes_set) %>%
    dplyr::group_by(gene, unit_group) %>%
    dplyr::summarise(mean_expr = mean(mean_expr, na.rm = TRUE),
                     .groups = "drop") %>%
    tidyr::pivot_wider(names_from  = unit_group,
                        values_from = mean_expr,
                        values_fill = 0) %>%
    tibble::column_to_rownames("gene")

  col_exist <- group_order[group_order %in% colnames(mat_r)]
  mat_r     <- mat_r[
    shared_genes_set[shared_genes_set %in% rownames(mat_r)],
    col_exist, drop = FALSE
  ]

  mat_z <- t(scale(t(mat_r)))
  mat_z[is.nan(mat_z)] <- 0
  mat_z <- pmin(pmax(mat_z, -2.5), 2.5)

  # 过滤：AD-Abeta 和 AD-Control 都 > Con-Control
  if (all(c("AD-Abeta","AD-Control","Con-Control") %in% colnames(mat_z))) {
    con_z      <- mat_z[, "Con-Control"]
    keep_genes <- rownames(mat_z)[
      mat_z[, "AD-Abeta"]   > con_z &
      mat_z[, "AD-Control"] > con_z
    ]
    message("  过滤前: ", nrow(mat_z), " 后: ", length(keep_genes))
    if (length(keep_genes) == 0) { message("跳过"); return(NULL) }
    mat_z <- mat_z[keep_genes, , drop = FALSE]
  }

  col_anno <- data.frame(
    Group     = col_exist,
    row.names = col_exist
  )
  anno_colors <- list(
    Group = c("Con-Control" = "#4DBBD5",
              "AD-Control"  = "#F39B7F",
              "AD-Abeta"    = "#E64B35")[col_exist]
  )

  draw_heatmap(
    mat_z       = mat_z,
    col_anno    = col_anno,
    anno_colors = anno_colors,
    title       = "Shared (CA1 + SLRM + FAS)",
    file_prefix = "NVU_unit_heatmap_Shared"
  )
}

# ════════════════════════════════════════════════════════════
# 单脑区热图（DESeq2 top genes，三组列）
# ════════════════════════════════════════════════════════════

make_region_heatmap <- function(region) {
  message("\n=== 脑区: ", region, " ===")

  genes_r <- top_per_region[[region]] %>%
    .[. %in% rownames(NVU_unit)]

  if (length(genes_r) == 0) { message("无有效基因"); return(NULL) }

  col_ids <- paste0(group_order, "\n", region)

  mat_r <- mean_expr_all %>%
    dplyr::filter(gene %in% genes_r, area_m == region) %>%
    mutate(col_id = paste0(unit_group, "\n", area_m)) %>%
    dplyr::select(gene, col_id, mean_expr) %>%
    tidyr::pivot_wider(names_from  = col_id,
                        values_from = mean_expr,
                        values_fill = 0) %>%
    tibble::column_to_rownames("gene")

  col_exist <- col_ids[col_ids %in% colnames(mat_r)]
  mat_r     <- mat_r[
    genes_r[genes_r %in% rownames(mat_r)],
    col_exist, drop = FALSE
  ]

  mat_z <- t(scale(t(mat_r)))
  mat_z[is.nan(mat_z)] <- 0
  mat_z <- pmin(pmax(mat_z, -2.5), 2.5)

  # 过滤：AD-Abeta >= AD-Control > Con-Control，且排除 shared
  abeta_col  <- paste0("AD-Abeta\n",    region)
  adctrl_col <- paste0("AD-Control\n",  region)
  con_col    <- paste0("Con-Control\n", region)

  if (all(c(abeta_col, adctrl_col, con_col) %in% colnames(mat_z))) {
    keep_cond1 <- mat_z[, abeta_col]  >= mat_z[, adctrl_col]
    keep_cond2 <- mat_z[, adctrl_col] >  mat_z[, con_col]
    keep_cond3 <- !rownames(mat_z) %in% shared_genes_set

    keep_genes <- rownames(mat_z)[keep_cond1 & keep_cond2 & keep_cond3]

    message("  候选: ", nrow(mat_z),
            "  AD-Abeta>=AD-Ctrl: ", sum(keep_cond1),
            "  AD-Ctrl>Con: ", sum(keep_cond2),
            "  排除shared后: ", length(keep_genes))

    if (length(keep_genes) == 0) { message("  -> 跳过"); return(NULL) }
    mat_z <- mat_z[keep_genes, , drop = FALSE]
  }

  col_anno <- data.frame(col_id = colnames(mat_z)) %>%
    tidyr::separate(col_id, into = c("Group","Region"),
                    sep = "\n", remove = FALSE) %>%
    tibble::column_to_rownames("col_id")

  anno_colors <- list(
    Group  = c("Con-Control" = "#4DBBD5",
               "AD-Control"  = "#F39B7F",
               "AD-Abeta"    = "#E64B35")[unique(col_anno$Group)],
    Region = c("CA1"  = "#3C5488",
               "SLRM" = "#00A087",
               "FAS"  = "#7E6148")[unique(col_anno$Region)]
  )

  draw_heatmap(
    mat_z       = mat_z,
    col_anno    = col_anno,
    anno_colors = anno_colors,
    title       = region,
    file_prefix = paste0("NVU_unit_heatmap_", region)
  )
}

# ════════════════════════════════════════════════════════════
# 运行
# ════════════════════════════════════════════════════════════

heatmap_shared <- make_shared_heatmap()
heatmap_CA1    <- make_region_heatmap("CA1")
heatmap_SLRM   <- make_region_heatmap("SLRM")
heatmap_FAS    <- make_region_heatmap("FAS")

message("\n完成: 四张热图已保存")

In [ ]:
# ════════════════════════════════════════════════════════════
# 修正 shared 基因集：
# 条件1：在三个脑区 DESeq2 up 结果中都出现
# 条件2：AD-Abeta z-score 是三组中最高
# 条件3：不出现在任何单脑区的 top30（纯 shared，无交集）
# ════════════════════════════════════════════════════════════

# ── 各脑区全部显著上调基因 ───────────────────────────────────
all_up_per_region <- lapply(region_order, function(r) {
  Genes %>%
    dplyr::filter(area_m == r, direction == "up") %>%
    dplyr::pull(gene) %>%
    unique()
})
names(all_up_per_region) <- region_order

message("各脑区上调基因数:")
print(sapply(all_up_per_region, length))

# 条件1：三个脑区都显著上调
shared_candidate <- Reduce(intersect, all_up_per_region) %>%
  .[. %in% rownames(NVU_unit)]
message("三脑区交集: ", length(shared_candidate), " 基因")

# 条件3：不出现在任何单脑区 top30（纯 shared）
# 单脑区 top30
top_per_region <- lapply(region_order, function(r) {
  Genes %>%
    dplyr::filter(area_m == r, direction == "up") %>%
    dplyr::arrange(desc(log2FoldChange)) %>%
    dplyr::slice_head(n = n_top) %>%
    dplyr::pull(gene)
})
names(top_per_region) <- region_order

# 单脑区特异基因（只在一个脑区 top30 出现）
region_specific_genes <- lapply(region_order, function(r) {
  other_regions <- setdiff(region_order, r)
  setdiff(top_per_region[[r]],
          unlist(top_per_region[other_regions]))
})
names(region_specific_genes) <- region_order

# 所有单脑区 top30 的并集（用于排除）
all_top30_union <- unlist(top_per_region) %>% unique()

# shared 候选：在三脑区都显著，且不在任何单脑区 top30
shared_pure <- setdiff(shared_candidate, all_top30_union) %>%
  .[. %in% rownames(NVU_unit)]

message("排除单脑区 top30 后 shared: ", length(shared_pure))

# 如果太少，放宽：不在任何单脑区特异基因里即可
if (length(shared_pure) < 5) {
  message("放宽条件：只排除单脑区特异基因（非 top30 全集）")
  all_specific_union <- unlist(region_specific_genes) %>% unique()
  shared_pure <- setdiff(shared_candidate, all_specific_union) %>%
    .[. %in% rownames(NVU_unit)]
  message("放宽后 shared: ", length(shared_pure))
}

message("最终 shared 基因: ")
print(shared_pure)

# ════════════════════════════════════════════════════════════
# shared 热图
# ════════════════════════════════════════════════════════════

make_shared_heatmap <- function(n_show = 30) {
  message("\n=== Shared 热图 ===")
  if (length(shared_pure) == 0) { message("无基因，跳过"); return(NULL) }

  # 按三脑区平均 log2FC 降序选 top n_show
  genes_show <- Genes %>%
    dplyr::filter(gene %in% shared_pure,
                  area_m %in% region_order) %>%
    dplyr::group_by(gene) %>%
    dplyr::summarise(mean_fc = mean(log2FoldChange, na.rm = TRUE),
                     .groups = "drop") %>%
    dplyr::arrange(desc(mean_fc)) %>%
    dplyr::slice_head(n = n_show) %>%
    dplyr::pull(gene) %>%
    .[. %in% rownames(NVU_unit)]

  message("展示基因数: ", length(genes_show))
  if (length(genes_show) == 0) { message("跳过"); return(NULL) }

  # 三脑区表达均值合并
  mat_r <- mean_expr_all %>%
    dplyr::filter(gene %in% genes_show) %>%
    dplyr::group_by(gene, unit_group) %>%
    dplyr::summarise(mean_expr = mean(mean_expr, na.rm = TRUE),
                     .groups = "drop") %>%
    tidyr::pivot_wider(names_from  = unit_group,
                        values_from = mean_expr,
                        values_fill = 0) %>%
    tibble::column_to_rownames("gene")

  col_exist <- group_order[group_order %in% colnames(mat_r)]
  mat_r     <- mat_r[genes_show[genes_show %in% rownames(mat_r)],
                     col_exist, drop = FALSE]

  mat_z <- t(scale(t(mat_r)))
  mat_z[is.nan(mat_z)] <- 0
  mat_z <- pmin(pmax(mat_z, -2.5), 2.5)

  # 条件2：AD-Abeta z-score 必须是三组中最高
  if ("AD-Abeta" %in% colnames(mat_z)) {
    row_max    <- apply(mat_z, 1, max)
    abeta_z    <- mat_z[, "AD-Abeta"]
    keep_genes <- rownames(mat_z)[abeta_z == row_max & abeta_z > 0]

    message("  AD-Abeta 最高过滤前: ", nrow(mat_z),
            "  后: ", length(keep_genes))

    if (length(keep_genes) == 0) {
      message("  放宽：只要 AD-Abeta > Con-Control")
      keep_genes <- rownames(mat_z)[
        mat_z[, "AD-Abeta"] > mat_z[, "Con-Control"]
      ]
      message("  放宽后: ", length(keep_genes))
    }

    if (length(keep_genes) == 0) { message("跳过"); return(NULL) }
    mat_z <- mat_z[keep_genes, , drop = FALSE]
  }

  message("  最终展示基因数: ", nrow(mat_z))

  col_anno <- data.frame(
    Group     = col_exist,
    row.names = col_exist
  )
  anno_colors <- list(
    Group = c("Con-Control" = "#4DBBD5",
              "AD-Control"  = "#F39B7F",
              "AD-Abeta"    = "#E64B35")[col_exist]
  )

  draw_heatmap(
    mat_z       = mat_z,
    col_anno    = col_anno,
    anno_colors = anno_colors,
    title       = "Shared up-regulated genes (CA1 + SLRM + FAS)",
    file_prefix = "NVU_unit_heatmap_Shared"
  )
}

# ════════════════════════════════════════════════════════════
# 单脑区热图（排除 shared_pure）
# ════════════════════════════════════════════════════════════

make_region_heatmap <- function(region) {
  message("\n=== 脑区: ", region, " ===")

  # 排除 shared_pure 基因
  genes_r <- top_per_region[[region]] %>%
    setdiff(shared_pure) %>%             # ← 排除 shared
    .[. %in% rownames(NVU_unit)]

  message("  排除 shared 后基因数: ", length(genes_r))
  if (length(genes_r) == 0) { message("无有效基因"); return(NULL) }

  col_ids <- paste0(group_order, "\n", region)

  mat_r <- mean_expr_all %>%
    dplyr::filter(gene %in% genes_r, area_m == region) %>%
    mutate(col_id = paste0(unit_group, "\n", area_m)) %>%
    dplyr::select(gene, col_id, mean_expr) %>%
    tidyr::pivot_wider(names_from  = col_id,
                        values_from = mean_expr,
                        values_fill = 0) %>%
    tibble::column_to_rownames("gene")

  col_exist <- col_ids[col_ids %in% colnames(mat_r)]
  mat_r     <- mat_r[
    genes_r[genes_r %in% rownames(mat_r)],
    col_exist, drop = FALSE
  ]

  mat_z <- t(scale(t(mat_r)))
  mat_z[is.nan(mat_z)] <- 0
  mat_z <- pmin(pmax(mat_z, -2.5), 2.5)

  # 过滤：AD-Abeta >= AD-Control > Con-Control
  abeta_col  <- paste0("AD-Abeta\n",    region)
  adctrl_col <- paste0("AD-Control\n",  region)
  con_col    <- paste0("Con-Control\n", region)

  if (all(c(abeta_col, adctrl_col, con_col) %in% colnames(mat_z))) {
    keep_cond1 <- mat_z[, abeta_col]  >= mat_z[, adctrl_col]
    keep_cond2 <- mat_z[, adctrl_col] >  mat_z[, con_col]
    keep_genes <- rownames(mat_z)[keep_cond1 & keep_cond2]

    message("  AD-Abeta>=AD-Ctrl: ", sum(keep_cond1),
            "  AD-Ctrl>Con: ", sum(keep_cond2),
            "  同时满足: ", length(keep_genes))

    if (length(keep_genes) == 0) { message("跳过"); return(NULL) }
    mat_z <- mat_z[keep_genes, , drop = FALSE]
  }

  col_anno <- data.frame(col_id = colnames(mat_z)) %>%
    tidyr::separate(col_id, into = c("Group","Region"),
                    sep = "\n", remove = FALSE) %>%
    tibble::column_to_rownames("col_id")

  anno_colors <- list(
    Group  = c("Con-Control" = "#4DBBD5",
               "AD-Control"  = "#F39B7F",
               "AD-Abeta"    = "#E64B35")[unique(col_anno$Group)],
    Region = c("CA1"  = "#3C5488",
               "SLRM" = "#00A087",
               "FAS"  = "#7E6148")[unique(col_anno$Region)]
  )

  draw_heatmap(
    mat_z       = mat_z,
    col_anno    = col_anno,
    anno_colors = anno_colors,
    title       = region,
    file_prefix = paste0("NVU_unit_heatmap_", region)
  )
}

# ════════════════════════════════════════════════════════════
# 运行
# ════════════════════════════════════════════════════════════

heatmap_shared <- make_shared_heatmap(n_show = 30)
heatmap_CA1    <- make_region_heatmap("CA1")
heatmap_SLRM   <- make_region_heatmap("SLRM")
heatmap_FAS    <- make_region_heatmap("FAS")

message("\n完成")

### 富集分析

In [ ]:
# ════════════════════════════════════════════════════════════
# GO 富集分析 —— AD-Abeta 上调基因（三个脑区）
# ═════════# 解决命名空间冲突

library(clusterProfiler)
library(org.Hs.eg.db)
library(ggplot2)
library(dplyr)
library(patchwork)
library(stringr)
select    <- dplyr::select
filter    <- dplyr::filter
mutate    <- dplyr::mutate
arrange   <- dplyr::arrange
summarise <- dplyr::summarise
group_by  <- dplyr::group_by
# ════════════════════════════════════════════════════════════
# 1. 准备基因列表
# ════════════════════════════════════════════════════════════

up_genes <- Genes %>%
  dplyr::filter(direction == "up") %>%
  select(gene, area_m, log2FoldChange, padj)

message("各脑区上调基因数:")
print(table(up_genes$area_m))

# SYMBOL → Entrez ID
entrez_list <- lapply(c("CA1","SLRM","FAS"), function(region) {
  message("转换 ", region, ":")
  genes_r <- up_genes %>%
    dplyr::filter(area_m == region) %>%
    pull(gene) %>% unique()
  df <- bitr(genes_r,
             fromType = "SYMBOL",
             toType   = "ENTREZID",
             OrgDb    = org.Hs.eg.db)
  message("  成功: ", nrow(df), "/", length(genes_r))
  df$area_m <- region
  df
})
names(entrez_list) <- c("CA1","SLRM","FAS")

# ════════════════════════════════════════════════════════════
# 2. GO 富集（BP + MF + CC 三个本体）
# ════════════════════════════════════════════════════════════

run_go <- function(region, ont = "BP") {
  gene_ids <- entrez_list[[region]]$ENTREZID
  if (length(gene_ids) < 5) return(NULL)
  tryCatch({
    enrichGO(
      gene          = gene_ids,
      OrgDb         = org.Hs.eg.db,
      ont           = ont,
      pAdjustMethod = "BH",
      pvalueCutoff  = 0.05,
      qvalueCutoff  = 0.2,
      readable      = TRUE
    )
  }, error = function(e) {
    message(region, " GO-", ont, " 失败: ", conditionMessage(e)); NULL
  })
}

message("\n=== GO BP ===")
go_BP <- lapply(c("CA1","SLRM","FAS"), run_go, ont = "BP")
names(go_BP) <- c("CA1","SLRM","FAS")

message("\n=== GO MF ===")
go_MF <- lapply(c("CA1","SLRM","FAS"), run_go, ont = "MF")
names(go_MF) <- c("CA1","SLRM","FAS")

message("\n=== GO CC ===")
go_CC <- lapply(c("CA1","SLRM","FAS"), run_go, ont = "CC")
names(go_CC) <- c("CA1","SLRM","FAS")

# ════════════════════════════════════════════════════════════
# 3. compareCluster（三脑区并排比较）
# ════════════════════════════════════════════════════════════

gene_list_compare <- lapply(c("CA1","SLRM","FAS"),
                             function(r) entrez_list[[r]]$ENTREZID)
names(gene_list_compare) <- c("CA1","SLRM","FAS")

run_compare <- function(ont) {
  tryCatch({
    compareCluster(
      geneClusters  = gene_list_compare,
      fun           = "enrichGO",
      OrgDb         = org.Hs.eg.db,
      ont           = ont,
      pAdjustMethod = "BH",
      pvalueCutoff  = 0.05,
      readable      = TRUE
    )
  }, error = function(e) {
    message("compareCluster GO-", ont, " 失败: ", conditionMessage(e)); NULL
  })
}

message("\n=== compareCluster ===")
compare_BP <- run_compare("BP")
compare_MF <- run_compare("MF")
compare_CC <- run_compare("CC")

# ════════════════════════════════════════════════════════════
# 4. 可视化
# ════════════════════════════════════════════════════════════

region_colors <- c("CA1"  = "#3C5488",
                   "SLRM" = "#00A087",
                   "FAS"  = "#7E6148")

# ── 单脑区 dotplot ────────────────────────────────────────────
make_dotplot <- function(res, region, ont, n_show = 15) {
  if (is.null(res) || nrow(res@result) == 0) return(NULL)

  df <- res@result %>%
    dplyr::filter(p.adjust < 0.05) %>%
    arrange(p.adjust) %>%
    slice_head(n = n_show) %>%
    mutate(
      Description   = str_wrap(Description, width = 45),
      GeneRatio_num = sapply(GeneRatio, function(x) {
        v <- as.numeric(strsplit(x, "/")[[1]])
        v[1] / v[2]
      })
    )

  if (nrow(df) == 0) return(NULL)

  ggplot(df, aes(x = GeneRatio_num,
                  y = reorder(Description, GeneRatio_num),
                  size = Count, color = p.adjust)) +
    geom_point(alpha = 0.88) +
    scale_color_gradient(low  = region_colors[[region]],
                         high = "grey85",
                         name = "adj. p") +
    scale_size_continuous(name = "Gene\ncount", range = c(2, 8)) +
    scale_x_continuous(expand = expansion(mult = c(0.05, 0.2))) +
    labs(
      title = paste0(region, "  |  GO-", ont),
      x = "Gene ratio", y = NULL
    ) +
    theme_classic(base_size = 11) +
    theme(
      plot.title      = element_text(face = "bold", size = 11,
                                      color = region_colors[[region]]),
      axis.text.y     = element_text(size = 8.5, color = "black"),
      axis.text.x     = element_text(size = 9,   color = "black"),
      axis.line       = element_line(linewidth = 0.4),
      axis.ticks      = element_line(linewidth = 0.35),
      legend.position = "right",
      legend.text     = element_text(size = 8),
      legend.title    = element_text(size = 9),
      plot.margin     = margin(6, 6, 6, 6)
    )
}

# ── compareCluster dotplot ────────────────────────────────────
make_compare_plot <- function(res, ont, n_show = 10) {
  if (is.null(res)) return(NULL)
  dotplot(res, showCategory = n_show) +
    scale_color_gradient(low = "#E64B35", high = "grey85",
                         name = "adj. p") +
    labs(
      title = paste0("GO-", ont, " comparison across regions"),
      x = NULL
    ) +
    theme_classic(base_size = 11) +
    theme(
      plot.title   = element_text(face = "bold", size = 12, hjust = 0.5),
      axis.text.x  = element_text(angle = 30, hjust = 1, size = 10,
                                   color = region_colors[
                                     levels(res@compareClusterResult$Cluster)
                                   ]),
      axis.text.y  = element_text(size = 9),
      legend.text  = element_text(size = 8),
      legend.title = element_text(size = 9),
      plot.margin  = margin(8, 8, 8, 8)
    )
}

# ════════════════════════════════════════════════════════════
# 5. 保存
# ════════════════════════════════════════════════════════════

save_plot <- function(p, filename, width = 10, height = 7) {
  if (is.null(p)) return(invisible(NULL))
  cairo_pdf(paste0(filename, ".pdf"), width = width, height = height)
  print(p); dev.off()
  ggsave(paste0(filename, ".png"), p,
         width = width, height = height, dpi = 300)
  message("  保存: ", filename)
}

# 单脑区：BP
for (region in c("CA1","SLRM","FAS")) {
  p <- make_dotplot(go_BP[[region]], region, "BP")
  if (!is.null(p)) {
    h <- max(5, nrow(go_BP[[region]]@result %>%
                       dplyr::filter(p.adjust < 0.05)) * 0.35 + 3)
    save_plot(p, paste0("GO_BP_", region), width = 9, height = h)
  }
}


# compareCluster
save_plot(make_compare_plot(compare_BP, "BP"),
          "GO_BP_compare", width = 12, height = 10,limitsize = FALSE)

# ── 三脑区拼图（BP）─────────────────────────────────────────
plots_bp <- lapply(c("CA1","SLRM","FAS"),
                    function(r) make_dotplot(go_BP[[r]], r, "BP", n_show=12))
plots_bp  <- plots_bp[!sapply(plots_bp, is.null)]

if (length(plots_bp) > 0) {
  fig_bp <- wrap_plots(plots_bp, ncol = length(plots_bp)) +
    plot_annotation(
      title   = expression("GO BP — up-regulated genes around A"*beta*" plaques"),
      caption = "BH-adjusted p < 0.05",
      theme   = theme(
        plot.title   = element_text(face = "bold", size = 13, hjust = 0.5),
        plot.caption = element_text(size = 8, color = "grey50")
      )
    )
  save_plot(fig_bp, "GO_BP_all_regions",
            width = 9 * length(plots_bp), height = 10)
}

# ════════════════════════════════════════════════════════════
# 6. 结果表
# ════════════════════════════════════════════════════════════

go_all_df <- lapply(c("BP","MF","CC"), function(ont) {
  go_list <- list(BP = go_BP, MF = go_MF, CC = go_CC)[[ont]]
  lapply(c("CA1","SLRM","FAS"), function(r) {
    res <- go_list[[r]]
    if (is.null(res)) return(NULL)
    res@result %>%
      dplyr::filter(p.adjust < 0.05) %>%
      mutate(area_m = r, ontology = ont)
  }) %>% bind_rows()
}) %>% bind_rows()

write.csv(go_all_df, "NVU_GO_all_results.csv", row.names = FALSE)

message("\n=== 完成 ===")
message("显著 GO 条目数:")
print(table(go_all_df$area_m, go_all_df$ontology))

In [ ]:
save_plot <- function(p, filename, width = 10, height = 7, limitsize = FALSE) {
  if (is.null(p)) return(invisible(NULL))

  # 限制最大尺寸，避免过大
  height <- min(height, 40)
  width  <- min(width,  40)

  cairo_pdf(paste0(filename, ".pdf"), width = width, height = height)
  print(p)
  dev.off()

  ggsave(paste0(filename, ".png"), p,
         width     = width,
         height    = height,
         dpi       = 300,
         limitsize = limitsize)

  message("  保存: ", filename, " (", round(width,1),
          " × ", round(height,1), " in)")
}

# 重新跑单脑区 BP 图
for (region in c("CA1","SLRM","FAS")) {
  p <- make_dotplot(go_BP[[region]], region, "BP")
  if (!is.null(p)) {
    n_sig <- go_BP[[region]]@result %>%
      dplyr::filter(p.adjust < 0.05) %>%
      nrow()
    h <- min(max(5, n_sig * 0.35 + 3), 40)
    save_plot(p, paste0("GO_BP_", region), width = 9, height = h)
  }
}

# 单脑区 MF
for (region in c("CA1","SLRM","FAS")) {
  p <- make_dotplot(go_MF[[region]], region, "MF")
  if (!is.null(p)) {
    n_sig <- go_MF[[region]]@result %>%
      dplyr::filter(p.adjust < 0.05) %>%
      nrow()
    h <- min(max(5, n_sig * 0.35 + 3), 40)
    save_plot(p, paste0("GO_MF_", region), width = 9, height = h)
  }
}

# compareCluster
save_plot(make_compare_plot(compare_BP, "BP"),
          "GO_BP_compare", width = 12, height = 10)
save_plot(make_compare_plot(compare_MF, "MF"),
          "GO_MF_compare", width = 12, height = 8)
save_plot(make_compare_plot(compare_CC, "CC"),
          "GO_CC_compare", width = 12, height = 8)

# 三脑区拼图
plots_bp <- lapply(c("CA1","SLRM","FAS"),
                    function(r) make_dotplot(go_BP[[r]], r, "BP", n_show = 12))
plots_bp <- plots_bp[!sapply(plots_bp, is.null)]

if (length(plots_bp) > 0) {
  fig_bp <- patchwork::wrap_plots(plots_bp, ncol = length(plots_bp)) +
    patchwork::plot_annotation(
      title   = expression("GO BP — up-regulated genes around A"*beta*" plaques"),
      caption = "BH-adjusted p < 0.05",
      theme   = theme(
        plot.title   = element_text(face = "bold", size = 13, hjust = 0.5),
        plot.caption = element_text(size = 8, color = "grey50")
      )
    )
  save_plot(fig_bp, "GO_BP_all_regions",
            width = min(9 * length(plots_bp), 40), height = 10)
}